# Data ingestion Layer:
- get documents
- create chunks
- embeddings
- call the llm transformer instance to generate our knowledge graph(KG)

#### **Load Libraries**

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
import ollama
import re # regex - regular experessions 
from langchain_ollama import ChatOllama
from langchain_neo4j import Neo4jGraph, LLMGraphTransformer
import nest_asyncio
nest_asyncio.apply()

import asyncio
import time
import numpy as np


/tmp/ipykernel_145558/1337507235.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


#### **Load Data**

In [5]:
loader = PyPDFLoader("data/Darwinism.pdf")
pages = loader.load()

In [6]:
pages

[Document(metadata={'producer': 'Skia/PDF m154', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/154.0.0.0 Safari/537.36', 'creationdate': '2026-10-16T07:09:35+00:00', 'title': 'Darwinism - Wikipedia', 'moddate': '2026-10-16T07:09:35+00:00', 'source': 'data/Darwinism.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='Charles Darwin in 1868\nDarwinism\nDarwinism is a term used to describe a theory of biological\nevolution developed by the English naturalist Charles Darwin\n(1809–1882) and his contemporaries. The theory states that\nall species of organisms arise and develop through the natural\nselection of small, inherited variations that increase the\nindividual\'s ability to compete, survive, and reproduce. Also\ncalled Darwinian theory, it originally included the broad\nconcepts of transmutation of species or of evolution which\ngained general scientific acceptance after Darwin published\nOn the Origin of Species in

In [7]:
# clean data
def cleanText(text):
    text = re.sub(r'-\n(\w)', r'\1', text)     # re-join words hyphenated across lines
    text = text.replace('•', '\n- ')           # bullets as their own lines
    text = re.sub(r'[ \t]+', ' ', text)        # collapse spaces/tabs only
    text = re.sub(r'\n{3,}', '\n\n', text)     # cap blank lines
    return text.strip()

#### **Split and create chunks**

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 800, chunk_overlap= 80) # created an instance of the text splitter
chunks = splitter.split_documents(pages)

In [9]:
# embedding function
def createEmbedding(text):
    response = ollama.embed(
    model='nomic-embed-text-v2-moe',
    input=text,
    )
    return response.embeddings[0]

In [10]:
doc_embeddings = [] # this is our list of embeddings from out document

for chunk in chunks:
    chunk_embedding = createEmbedding(chunk.page_content)
    doc_embeddings.append(chunk_embedding) 

In [11]:
with open("doc_embeddings_two.txt", "w") as f:
    for item in doc_embeddings:
        f.write(f"{item}\n")

In [12]:
# creates an instance of the model we will use low temperature and high content window and adding num_predict reduces how much the model generates per call, 
llama_llm =  ChatOllama(model="llama3.2:3b",temperature=0, num_predict=800) 

In [13]:
# connect to neo4J DB
graph = Neo4jGraph(url="neo4j://127.0.0.1:7687", username="neo4j", password="neo4j123")

In [14]:

# transformer init: The help in attaining the node + relationships  in the dta hence creating our knowledge graph's
llama_transformer = LLMGraphTransformer(llm=llama_llm) 

#### **create knowledge graph**

In [15]:
len(chunks)

38

In [16]:
import time
start = time.time()

graph_documents = await llama_transformer.aconvert_to_graph_documents(chunks[:4])   


print(f"Took {time.time() - start:.1f}s")
print(f"Got {len(graph_documents)} graph documents")

Took 462.8s
Got 4 graph documents


In [17]:
graph_documents

[GraphDocument(nodes=[], relationships=[], source=Document(metadata={'producer': 'Skia/PDF m154', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/154.0.0.0 Safari/537.36', 'creationdate': '2026-10-16T07:09:35+00:00', 'title': 'Darwinism - Wikipedia', 'moddate': '2026-10-16T07:09:35+00:00', 'source': 'data/Darwinism.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content="Charles Darwin in 1868\nDarwinism\nDarwinism is a term used to describe a theory of biological\nevolution developed by the English naturalist Charles Darwin\n(1809–1882) and his contemporaries. The theory states that\nall species of organisms arise and develop through the natural\nselection of small, inherited variations that increase the\nindividual's ability to compete, survive, and reproduce. Also\ncalled Darwinian theory, it originally included the broad\nconcepts of transmutation of species or of evolution which\ngained general scientific acceptance aft

In [18]:
with open("graph_doc.txt", "w") as f:
    for item in graph_documents:
        f.write(f"{item}\n")

Adds the Knowledge graph to our Neo4J Graph DB

In [19]:
for graph_doc in graph_documents: 
    graph.add_graph_documents([graph_doc], baseEntityLabel=True, include_source=True) # adds the knowledge graph to DB

In [20]:

def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# --- Vector side (reuses doc_embeddings + chunks from earlier) ---
def vector_retrieve(question, chunks, doc_embeddings, top_k=3):
    q_vec = createEmbedding(question)
    scored = [(cosine_sim(q_vec, v), c) for v, c in zip(doc_embeddings, chunks)]
    scored.sort(key=lambda x: x[0], reverse=True)
    return [c.page_content for _, c in scored[:top_k]]

def graph_retrieve_facts(question, graph, model="llama3.2"):
    """Returns raw facts (a list), no generation — for use inside hybrid_rag_answer."""
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": "What is the single main entity (person, place, or concept) this question is about? Reply with just the name, nothing else."},
            {"role": "user", "content": question},
        ],
        options={"temperature": 0},
    )
    entity = response["message"]["content"].strip()

    facts = graph.query(
        """
        MATCH (n)-[r]-(m)
        WHERE toLower(n.id) CONTAINS toLower($name)
        RETURN n.id AS entity, type(r) AS relation, m.id AS related
        LIMIT 20
        """,
        {"name": entity}
    )
    return [f"{f['entity']} -{f['relation']}-> {f['related']}" for f in facts]

def hybrid_rag_answer(question, chunks, doc_embeddings, graph, model="llama3.2"):
    vector_context = vector_retrieve(question, chunks, doc_embeddings)
    graph_context = graph_retrieve_facts(question, graph, model=model)   # ← changed here

    combined_context = f"""Text passages:
{chr(10).join(vector_context)}

Graph facts:
{chr(10).join(graph_context) if graph_context else "(none found)"}"""

    prompt = f"""Answer the question using the context below.

Context:
{combined_context}

Question: {question}"""

    response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
    return response["message"]["content"]

In [21]:
print(graph_retrieve_facts("What is Darwinian theory?", graph=graph, model="deepseek-r1:1.5b"))

[]


In [25]:
print(graph_retrieve_facts("Who is Huxley?", graph=graph, model="llama3.2:3b"))

[]


In [23]:
print(graph_retrieve_facts("Who is Huxley?", graph=graph, model="mistral:7b"))

['Huxley -MENTIONS-> 3d887d71730db3c53c884c1c99c6d18f', 'Huxley -COINED-> Darwinism', 'Huxley -RELATED-> August Weismann', 'Huxley -RELATED-> Lamarckism', 'Huxley -RELATED-> Neo-Darwinian Theories', 'Huxley -RELATED-> English Biologist']


In [26]:
# the above leaks the chunk id hence: 
def graph_retrieve_facts_clean(question, graph, model="llama3.2"):
    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": "What is the single main entity (person, place, or concept) this question is about? Reply with just the name, nothing else."},
            {"role": "user", "content": question},
        ],
        options={"temperature": 0},
    )
    entity = response["message"]["content"].strip()

    facts = graph.query(
        """
        MATCH (n)-[r]-(m)
        WHERE toLower(n.id) CONTAINS toLower($name)
        RETURN n.id AS entity, type(r) AS relation, m.id AS related
        LIMIT 20
        """,
        {"name": entity}
    )

    # filter out MENTIONS edges to chunk-ID nodes (32-char hex strings)
    import re
    clean_facts = [
        f"{f['entity']} -{f['relation']}-> {f['related']}"
        for f in facts
        if not re.match(r'^[0-9a-f]{32}$', f['related'])
    ]
    return clean_facts

In [28]:
print(graph_retrieve_facts_clean("Who is Huxley?", graph=graph, model="mistral:7b"))

['Huxley -COINED-> Darwinism', 'Huxley -RELATED-> August Weismann', 'Huxley -RELATED-> Lamarckism', 'Huxley -RELATED-> Neo-Darwinian Theories', 'Huxley -RELATED-> English Biologist']


In [24]:
print(hybrid_rag_answer("Who is Huxley and what did he say about Darwin?", chunks, doc_embeddings, graph))

According to the context, Thomas Henry Huxley was an English biologist. When he first read Darwin's theory in 1858, he exclaimed, "How extremely stupid not to have thought of that!"
